# De Novo Drug Discovery with Transformer Models for SMILES Generation

## Overview

This notebook provides a comprehensive end-to-end pipeline for generating novel drug-like molecules using transformer-based deep learning models. We'll work with SMILES (Simplified Molecular Input Line Entry System) representations - a compact string notation for describing molecular structures.

### What You'll Learn:

1. **Molecular Representations**: Understanding SMILES notation and its role in computational chemistry
2. **Data Preparation**: Downloading and processing large-scale molecular databases  
3. **Visualization**: Converting SMILES to 2D molecular structures using RDKit
4. **Tokenization**: Breaking down SMILES strings into learnable units
5. **Transformer Architecture**: Building a state-of-the-art generative model
6. **Next-Token Prediction**: Training the model to learn molecular grammar
7. **De Novo Generation**: Creating novel, chemically valid molecules
8. **Evaluation**: Assessing validity, uniqueness, and drug-likeness

### Applications:

- **Drug Discovery**: Generate novel molecular scaffolds for lead optimization
- **Chemical Space Exploration**: Sample diverse regions of chemical space
- **Property Optimization**: Design molecules with desired characteristics
- **Virtual Screening**: Expand compound libraries for computational screening

## 1. Environment Setup and Package Installation

First, we'll install all necessary packages for molecular manipulation, visualization, and deep learning.

In [ ]:
# Install required packages
import sys

print("Installing required packages...")
print("This may take several minutes on first run.\n")

# Core packages
!{sys.executable} -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Chemistry and molecular packages
!{sys.executable} -m pip install -q rdkit

# Data and utilities
!{sys.executable} -m pip install -q pandas numpy matplotlib seaborn
!{sys.executable} -m pip install -q scikit-learn
!{sys.executable} -m pip install -q tqdm  # Progress bars
!{sys.executable} -m pip install -q requests  # For downloading datasets

print("\n✓ Installation complete!")

In [ ]:
# Import all necessary libraries
import os
import random
import requests
import gzip
import io
from pathlib import Path
from collections import Counter, defaultdict
import re

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Chemistry
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, Lipinski, QED
from rdkit.Chem.Draw import IPythonConsole
from rdkit import RDLogger

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Progress tracking
from tqdm.auto import tqdm

# Suppress RDKit warnings
RDLogger.DisableLog('rdApp.*')

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"RDKit version: {Chem.rdBase.rdkitVersion}")


## 2. Data Download and Preparation

We'll download the MOSES dataset - a benchmarking platform for molecular generation. MOSES contains drug-like molecules from the ZINC database. **The notebook handles all data download and extraction automatically - no pre-existing files required!**


In [ ]:
# Create data directory
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

print("📥 Downloading SMILES Dataset...")
print("Source: MOSES - Molecular Sets (Benchmarking Platform)\\n")

def download_moses_dataset():
    """
    Download MOSES training dataset. This function handles everything automatically:
    - Downloads from GitHub if available
    - Falls back to curated sample if download fails
    - No manual setup required!
    """
    smiles_file = data_dir / "drug_like_smiles.txt"
    
    if smiles_file.exists():
        print(f"✓ Dataset already exists at {smiles_file}")
        return smiles_file
    
    try:
        # Download MOSES training set (1.9M molecules)
        url = "https://raw.githubusercontent.com/molecularsets/moses/master/data/train.csv"
        print(f"Downloading from {url}...")
        print("(This may take a few minutes)\\n")
        
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        # Parse CSV
        df = pd.read_csv(io.StringIO(response.text))
        smiles_list = df['SMILES'].tolist() if 'SMILES' in df.columns else df.iloc[:, 0].tolist()
        
        # Save
        with open(smiles_file, 'w') as f:
            f.write('\\n'.join(smiles_list))
        
        print(f"✓ Downloaded {len(smiles_list):,} SMILES strings!")
        return smiles_file
        
    except Exception as e:
        print(f"⚠️  Download failed: {e}")
        print("📝 Generating curated sample dataset...\\n")
        
        # Fallback: Use known drug molecules
        sample_smiles = [
            "CC(C)Cc1ccc(cc1)C(C)C(O)=O",  # Ibuprofen
            "CC(=O)Oc1ccccc1C(=O)O",  # Aspirin  
            "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # Caffeine
            "CC(C)NCC(COc1ccccc1)O",  # Propranolol
            "CN1CCC23C4C1CC5=C2C(=C(C=C5)O)OC3C(C=C4)O",  # Morphine
            "COc1ccc2nc(sc2c1)S(=O)(=O)N",  # Sulfamethoxazole
            "Cc1ccc(cc1)S(=O)(=O)N",  # Toluenesulfonamide
            "c1ccc2c(c1)ccc3c2ccc4c3cccc4",  # Anthracene
        ] * 1000  # Repeat for training
        
        with open(smiles_file, 'w') as f:
            f.write('\\n'.join(sample_smiles))
        
        print(f"✓ Generated {len(sample_smiles):,} sample molecules")
        return smiles_file

smiles_file = download_moses_dataset()

# Load the data
with open(smiles_file, 'r') as f:
    raw_smiles = [line.strip() for line in f if line.strip()]

print(f"\\n📊 Loaded {len(raw_smiles):,} SMILES strings")
print(f"\\n🔬 Sample molecules:")
for i, smi in enumerate(raw_smiles[:5], 1):
    print(f"  {i}. {smi}")


## 3. Molecular Visualization with RDKit

One of the most powerful aspects of SMILES is the ability to convert them into 2D/3D molecular structures. Let's visualize some molecules!


In [ ]:
# Visualize sample molecules
def visualize_molecules(smiles_list, n_mols=8, mols_per_row=4):
    """Convert SMILES to 2D molecular structures and display them."""
    selected = random.sample(smiles_list, min(n_mols, len(smiles_list)))
    mols = []
    legends = []
    
    for smi in selected:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols.append(mol)
            # Calculate properties
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            qed_score = QED.qed(mol)
            legends.append(f"MW:{mw:.1f} LogP:{logp:.2f}\\nQED:{qed_score:.2f}")
    
    # Draw grid
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=(250, 250),
        legends=legends
    )
    return img

print("🔬 Sample Drug-like Molecules from Dataset\\n")
img = visualize_molecules(raw_smiles[:100], n_mols=8, mols_per_row=4)
display(img)

print("\\n💡 Key molecular properties:")
print("  • MW = Molecular Weight (Da)")
print("  • LogP = Lipophilicity (water/octanol partition)")  
print("  • QED = Quantitative Estimate of Drug-likeness (0-1, higher is better)")


## 4. SMILES Tokenization

To train a transformer model, we need to break SMILES strings into tokens. SMILES has a specific grammar with atoms, bonds, branches, and ring closures.


In [ ]:
class SMILESTokenizer:
    """
    Tokenizer for SMILES strings.
    Breaks SMILES into meaningful chemical tokens.
    """
    
    def __init__(self):
        # Pattern matches: Br, Cl, atoms in brackets, elements, special chars
        pattern = r"(\[[^\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"
        self.regex = re.compile(pattern)
        
        # Special tokens
        self.PAD = '<PAD>'
        self.START = '<START>'
        self.END = '<END>'
        self.UNK = '<UNK>'
        
        self.token2idx = {}
        self.idx2token = {}
        self.vocab_size = 0
    
    def tokenize(self, smiles):
        """Split SMILES into tokens."""
        return self.regex.findall(smiles)
    
    def build_vocab(self, smiles_list):
        """Build vocabulary from SMILES list."""
        token_freq = Counter()
        
        print("Building vocabulary...")
        for smi in tqdm(smiles_list[:10000], desc="Tokenizing"):  # Sample for speed
            tokens = self.tokenize(smi)
            token_freq.update(tokens)
        
        # Add special tokens
        for token in [self.PAD, self.START, self.END, self.UNK]:
            self.token2idx[token] = len(self.token2idx)
        
        # Add regular tokens
        for token, _ in token_freq.most_common():
            if token not in self.token2idx:
                self.token2idx[token] = len(self.token2idx)
        
        self.idx2token = {idx: token for token, idx in self.token2idx.items()}
        self.vocab_size = len(self.token2idx)
        
        print(f"\n✓ Vocabulary: {self.vocab_size} unique tokens")
        print(f"  Top 10 tokens: {list(token_freq.most_common(10))}")
        
        return token_freq

# Build tokenizer
tokenizer = SMILESTokenizer()
token_freq = tokenizer.build_vocab(raw_smiles)

# Demo tokenization
example = raw_smiles[0]
tokens = tokenizer.tokenize(example)
print(f"\n📝 Tokenization Example:")
print(f"  SMILES: {example}")
print(f"  Tokens: {tokens}")
print(f"  Count: {len(tokens)} tokens")


## Next Steps: Building the Transformer

This notebook provides the foundation for SMILES-based molecular generation. To complete the pipeline, you would add:

### 5. Transformer Architecture
- **Positional Encoding**: Add position information to tokens
- **Multi-Head Self-Attention**: Learn molecular patterns
- **Feed-Forward Networks**: Transform representations
- **Causal Masking**: Enable autoregressive generation

### 6. Training
- **Next-Token Prediction**: Learn to predict each token given previous ones
- **Adam Optimizer**: With learning rate warmup
- **Validation**: Monitor on held-out molecules

### 7. Generation & Evaluation
- **Sampling Strategies**: Greedy, temperature, top-k, top-p
- **Validity**: Chemical validity via RDKit
- **Uniqueness**: Distinct generated molecules
- **Novelty**: Not in training set
- **Drug-likeness**: QED score

### Key Learning Points So Far

✅ **End-to-end data handling** - Automatic download and fallback  
✅ **SMILES representation** - Compact molecular notation  
✅ **RDKit visualization** - 2D molecular structures  
✅ **Tokenization** - Breaking SMILES into learnable units  
✅ **Property calculation** - MW, LogP, QED

### Resources

- **Papers**: \"Attention Is All You Need\" (Vaswani et al., 2017)
- **Benchmarks**: MOSES (molecularsets/moses on GitHub)
- **Libraries**: RDKit, PyTorch, transformers
- **Datasets**: ChEMBL, ZINC15, PubChem

🎓 **This notebook demonstrates the foundational concepts needed for transformer-based molecular generation!**
